# 1. INTRODUCTION
<center>
<img src="https://lumiere-a.akamaihd.net/v1/images/pp_findingnemo_herobanner_19752_eb5648d2.jpeg?region=0,0,2048,878" width=1300 height=1000 />
</center>



<font size="4"> **PROBLEM DESCRIPTION: Dissolved oxygen prediction in river water** </font>

<font size="3"> **This is a different type of competition!** Instead of submitting predictions, the task is to submit a **dataset** that will be used to train a random forest regressor model(provided). This model will then be used to make predictions against a hidden test dataset. The score will be the Mean Absolute Error (MAE) between the model predictions and ground truth of the test set.</font>

**ABOUT DATA**

<font size="3">The dataset has five indicators that are measured at 8 stations of the state water monitoring system.</font>

<font size="3">Indicators of river water quality in this dataset are:</font>
1. **O2_(i)**: Dissolved oxygen (O2) is measured in mgO2/cub. dm (ie milligrams of oxygen (O2) in the cubic decimeter);
2. **NH4_(i)**: Ammonium ions (NH4) concentration is measured in mg/cub. dm (ie milligrams in the cubic decimeter);
3. **NO2_(i)**: Nitrite ions (NO2) concentration is measured in mg/cub. dm (ie milligrams in the cubic decimeter);
4. **NO3_(i)**: Nitrate ions (NO3) concentration is measured in mg/cub. dm (ie milligrams in the cubic decimeter);
5. **BOD5_(i)**: Biochemical oxygen demand, which is determined in 5 days ("BOD5" or "BOD"). BOD5 is measured in mgO/cub. dm (ie milligrams of oxygen in the cubic decimeter).

i is the station number

**METRIC** ROOT MEAN SQUARED ERROR


**APPROACH**
1. <font size="3">Usage of original data set would be a good option regardless of LB score, I'm gonna go with MY CV since slightest changes in the appraoch can alter score massively</font>
2. <font size="3">Let's find out the feature importance based on entire dataset</font>
3. <font size="3">Then, five anomaly detection methods including OneClass SVM, Isolation Forest, Local Outlier Factor, Autoencoders, & PCA will be used to build some strategy to eliminate datapoints.</font>
4. <font size="3">After datapoints eliminations, let us now check the feature importance again and see which ones are important. This will exactly tell us the features that were contributing to outliers</font>
5. <font size="3">Eliminate unimportant columns by making them zeros</font>
6. <font size="3">Use test data without missing rows and use better models like LightGBM to predict target and add this to our submission</font>

# 2. IMPORTS

In [ ]:
import sys
assert sys.version_info >= (3, 5)

import sklearn
assert sklearn.__version__ >= "0.20"
import numpy as np
import os

import pandas as pd
import matplotlib.pyplot as plt
import missingno as msno
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_notebook
tqdm_notebook.get_lock().locks = []
from prettytable import PrettyTable
%matplotlib inline
import seaborn as sns
sns.set(style='darkgrid', font_scale=1.4)
from copy import deepcopy
from functools import partial
from itertools import combinations

from sklearn.cluster import KMeans
!pip install yellowbrick
from yellowbrick.cluster import KElbowVisualizer
import folium
from haversine import haversine
import random
from random import uniform
import gc
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn import metrics
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xg
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_squared_log_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler,PowerTransformer, FunctionTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from scipy.spatial.distance import mahalanobis
from scipy.stats import chi2
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy import stats
import statsmodels.api as sm
import math
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.base import BaseEstimator, TransformerMixin
!pip install optuna
import optuna
import xgboost as xgb
!pip install catboost
!pip install lightgbm --install-option=--gpu --install-option="--boost-root=C:/local/boost_1_69_0" --install-option="--boost-librarydir=C:/local/boost_1_69_0/lib64-msvc-14.1"
import lightgbm as lgb
!pip install category_encoders
from category_encoders import OneHotEncoder, OrdinalEncoder, CountEncoder, CatBoostEncoder
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import PassiveAggressiveRegressor, ARDRegression, RidgeCV, ElasticNetCV
from sklearn.linear_model import TheilSenRegressor, RANSACRegressor, HuberRegressor
from sklearn.ensemble import HistGradientBoostingRegressor,ExtraTreesRegressor,GradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from catboost import CatBoost, CatBoostRegressor,CatBoostClassifier
from catboost import Pool
from sklearn.neighbors import KNeighborsRegressor
# Suppress warnings
import warnings
warnings.filterwarnings("ignore")
pd.pandas.set_option('display.max_columns',None)

In [ ]:
data=pd.read_csv("/kaggle/input/playground-series-s3e21/sample_submission.csv")

orig_train=pd.read_csv("/kaggle/input/dissolved-oxygen-prediction-in-river-water/train.csv")
orig_test=pd.read_csv("/kaggle/input/dissolved-oxygen-prediction-in-river-water/test.csv")

orig_train=orig_train.rename(columns={"Id":"id"})
orig_test=orig_test.rename(columns={"Id":"id"})

data.head()

In [ ]:
for col in data.columns:
    data[col]=np.where(data[col]<0,0,data[col]) #observed negative values in the data

## 2.1 Missing Value Checks

In [ ]:
table = PrettyTable()

table.field_names = ['Column Name', 'Data Type', "Mising %", 'Original Missing %']
for column in data.columns:
    data_type = str(data[column].dtype)
    non_null_count_train= np.round(100-orig_train[column].count()/orig_train.shape[0]*100,1)
    non_null_count_data = np.round(100-data[column].count()/data.shape[0]*100,1)
    table.add_row([column, data_type,non_null_count_data, non_null_count_train])
print(table)

<font size="3">The original dataset has many missing values, especially from the third station. Which means, the synthetic data provided would be based on some imputation and only the first two stations have reliable data.</font>

In [ ]:
orig_train=orig_train.dropna()
data=pd.concat([data,orig_train],axis="rows").reset_index(drop=True)
data_pure=data.copy()

# 3. Feature Importance- With Outliers

<font size="3"> Let's get the Feature Importances of all the columns and do a forward selection to find the optimum columns that can minimize RMSE score</font>

In [ ]:
def rmse(y1,y2):
    return(np.sqrt(mean_squared_error(y1,y2)))
def get_most_important_features(data_modified, data_unmodified):
    
    X_train = data_modified.drop(columns=['target'])  
    y_train = data_modified['target'] 
    
    X_train_pure=data_unmodified.drop(columns=['target'])  
    y_train_pure=data_unmodified['target'] 
    
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    MAE = []
    feature_importances_list = []
    
    model = RandomForestRegressor(n_estimators=1000, max_depth=7, n_jobs=-1, random_state=42)

    for train_idx, val_idx in kfold.split(X_train):
        X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train_pure.iloc[val_idx]
        y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train_pure.iloc[val_idx]
        model.fit(X_train_fold, y_train_fold)

        y_pred = model.predict(X_val_fold)

        mae = rmse(y_val_fold, y_pred)
        MAE.append(mae)

        feature_importances = model.feature_importances_
        feature_importances_list.append(feature_importances)

    avg_mae = np.mean(MAE)

    # Calculate average feature importances over all folds
    avg_feature_importances = np.mean(feature_importances_list, axis=0)

    feature_importance_list = [(X_train.columns[i], importance) for i, importance in enumerate(avg_feature_importances)]

    sorted_features = sorted(feature_importance_list, key=lambda x: x[1], reverse=True)
    list_features=[feature[0] for feature in sorted_features]
    
    feature_names, importances = zip(*sorted_features)

    plt.figure(figsize=(8, 12))
    plt.barh(range(len(feature_names)), importances, color='green')
    plt.yticks(range(len(feature_names)), feature_names, fontsize=12)
    plt.xlabel('Average Feature Importance', fontsize=14)
    plt.ylabel('Features', fontsize=14)
    plt.title(f'Average Feature Importances with best RMSE score {avg_mae}', fontsize=16)
    plt.gca().invert_yaxis()  # Invert y-axis to have the most important feature on top
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    # Add data labels on the bars
    for index, value in enumerate(importances):
        plt.text(value + 0.005, index, f'{value:.3f}', fontsize=12, va='center')

    plt.tight_layout()
    plt.show()

    return list_features, avg_mae

list_features,best_score = get_most_important_features(data, data)
print(list_features)

<font size="3">The idea here is to add columns from the most important feature nd check if the best CV score seen so far decreases. As soon as we don't see an improvement, we stop</font>


In [ ]:
def feature_selection(data_modified,data_unmodified,list_features):
    random_seed = random.randint(0, 9999)
    
    X_train = data_modified.drop(columns=['target'])  
    y_train = data_modified['target'] 
    
    X_train_pure=data_unmodified.drop(columns=['target'])  
    y_train_pure=data_unmodified['target'] 
  
    model = RandomForestRegressor(n_estimators=1000, max_depth=7, n_jobs=-1, random_state=42) #reduced the estimators to make the process faster
    previous_rmse= float('inf')
    for i in range(1,len(list_features)):
        X_train_selected=X_train[list_features[:i]]
        X_train_pure_selected=X_train_pure[list_features[:i]]
        RMSE = []
        kfold = KFold(n_splits=5, shuffle=True, random_state=random_seed)
        for train_idx, val_idx in kfold.split(X_train_selected):
            X_train_fold, X_val_fold = X_train_selected.iloc[train_idx], X_train_pure_selected.iloc[val_idx]
            y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train_pure.iloc[val_idx]

            model.fit(X_train_fold, y_train_fold)

            y_pred = model.predict(X_val_fold)

            mae = rmse(y_val_fold, y_pred)
            RMSE.append(mae)
        rmse_combined=np.mean(RMSE)
        if rmse_combined<=previous_rmse:
            print(f"score {rmse_combined} improved by adding a feature") 
        else:
            print(f"No further improvement by adding columns, consider {i} features")
            n=i
            break
        previous_rmse=rmse_combined
    return n

In [ ]:
pre_anomaly_best_features=list_features[:feature_selection(data_pure,data_pure, list_features)]
print(f"Important features before removal of anomalies are {pre_anomaly_best_features}")

### Unbiased Scoring Function

In [ ]:
main_data=data_pure.copy()

def score_check_unbiased(data_modified, data_unmodified, n_estimators):
    random_seed = random.randint(0, 9999)
    X_train = data_modified.drop(columns=['target'])  
    y_train = data_modified['target'] 
    
    X_train_pure=data_unmodified.drop(columns=['target'])  
    y_train_pure=data_unmodified['target'] 
    model = RandomForestRegressor(n_estimators=n_estimators, max_depth=7, n_jobs=-1, random_state=42) 
    previous_rmse= float('inf')
    
    RMSE = []
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, val_idx in kfold.split(X_train):
        X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train_pure.iloc[val_idx]
        y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train_pure.iloc[val_idx]
        
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)

        mae = rmse(y_val_fold, y_pred)
        RMSE.append(mae)
    return np.mean(RMSE)
        
    

# 4. Anomaly Detection Methods

# 4.1 One Class SVM 
<font size="3">**One-Class SVM** is a machine learning algorithm used for anomaly detection. It's particularly useful when you have a dataset with predominantly normal data points and a few outliers or anomalies. The goal of One-Class SVM is to create a boundary that encompasses the majority of the normal data points, effectively identifying regions where anomalies are unlikely to reside</font>

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.linear_model import SGDOneClassSVM

def one_class_SVM(data_pure):
    data=data_pure.copy()
    model= SGDOneClassSVM(nu=0.6016924877965066)
    model.fit(data_pure.drop(columns=['id']))

    predicted_labels = model.predict(data_pure.drop(columns=['id']))

    # Convert predicted labels to identify outliers (predicted as -1)
    outliers = predicted_labels == -1

    data['outlier_One_SVM'] = outliers
    # Print the identified outliers
    identified_outliers = data[data['outlier_One_SVM']]
    print(f"Number of detected Potential outliers: {identified_outliers.shape[0]}")
    
    data_clean=data[~data['outlier_One_SVM']][list_features+['target']]
    print(f"RMSE Score without the identified outliers is {score_check_unbiased(data_clean,data_pure[list_features+['target']],250)}")
    
    return data
data= one_class_SVM(data_pure)

### Hyperparameter Tuning

In [ ]:
def objective(trial,data_pure):
    data=data_pure.copy()
    nu = trial.suggest_float('nu', 0.01, 1.0)  #
    model = SGDOneClassSVM(nu=nu)
    model.fit(main_data.drop(columns=['id']))

    predicted_labels = model.predict(main_data.drop(columns=['id']))

    # Convert predicted labels to identify outliers (predicted as -1)
    outliers = predicted_labels == -1

    data['outlier_One_SVM'] = outliers

    # Compute the score using score_check_unbiased
    data_clean = data[~data['outlier_One_SVM']]
    data_clean=data_clean.drop(columns=["outlier_One_SVM"])
    score = score_check_unbiased(data_clean, data_pure, 25)

    return score

# study = optuna.create_study(direction='minimize')

# study.optimize(lambda trial: objective(trial,data_pure), n_trials=100)  

# best_params = study.best_params
# best_nu = best_params['nu']

# best_model = SGDOneClassSVM(nu=best_nu)
# best_model.fit(data_pure.drop(columns=['id']))



# 4.2 Isolation Forest

<font size="3">**Isolation Forests** is an anomaly detection algorithm that operates by isolating anomalies in the data through a process of recursive partitioning. It works particularly well for high-dimensional data and can efficiently identify outliers without requiring a lot of computation</font>

In [ ]:
from sklearn.ensemble import IsolationForest

def isolation_forest(data_pure,data):
    model = IsolationForest(contamination=0.010650427383702971, random_state=0)

    model.fit(data_pure.drop(columns=['id']))

    # Predict the anomaly scores for each data point
    anomalies = model.predict(data_pure.drop(columns=['id']))

    outliers = anomalies == -1

    # Combine the outlier information with the original data and labels
    data['outlier_ISF'] = outliers

    # Print the identified outliers
    identified_outliers = data[data['outlier_ISF']]
    print(f"Number of detected Potential outliers: {identified_outliers.shape[0]}")
    
    data_clean=data[~data['outlier_ISF']][list_features+['target']]
    print(data_clean.shape)
    print(f"RMSE Score after without the identified outliers is {score_check_unbiased(data_clean,data_pure[list_features+['target']], 250)}")
    
    return data
data=isolation_forest(data_pure,data)

### BOOM! Eliminating 38 points improved RMSE massively. 
<font size="3"> This has been achieved by hyperparameter tuning</font>

In [ ]:
from sklearn.ensemble import IsolationForest
import optuna

def objective(trial, data_pure):
    data = data_pure.copy()
    contamination = trial.suggest_float('contamination', 0.01, 0.5)  # Tune the 'contamination' hyperparameter within a range
    model = IsolationForest(contamination=contamination, random_state=0)
    model.fit(main_data.drop(columns=['id']))

    predicted_labels = model.predict(main_data.drop(columns=['id']))

    outliers = predicted_labels == -1

    data['outlier_Isolation_Forest'] = outliers

    data_clean = data[~data['outlier_Isolation_Forest']]
    data_clean = data_clean.drop(columns=["outlier_Isolation_Forest"])
    score = score_check_unbiased(data_clean, data_pure, 25)

    return score

# study = optuna.create_study(direction='minimize')

# study.optimize(lambda trial: objective(trial, data_pure), n_trials=100)

# best_params = study.best_params
# best_contamination = best_params['contamination']

# best_model = IsolationForest(contamination=best_contamination, random_state=0)
# best_model.fit(data_pure.drop(columns=['id']))


# 4.3 Autoencoders

<font size="3">**Autoencoders** are a type of neural network architecture that can be used for various tasks, including anomaly detection. The basic idea behind autoencoders is to learn a compact representation (encoding) of the input data and then reconstruct the original data from this encoding. In the context of anomaly detection, we exploit the fact that autoencoders struggle to accurately reconstruct anomalous or out-of-distribution data, which allows us to identify outliers</font>

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models

def autoencoder_anomaly(main_data,data):

    features=main_data.drop(columns=['id'])

#     scaler = StandardScaler()
    scaled_features =features

    # Build the autoencoder architecture
    input_dim = scaled_features.shape[1]
    encoding_dim = 32  # Adjust this based on the complexity of your data

    autoencoder = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(encoding_dim, activation='relu'),
        layers.Dense(input_dim, activation='linear')  # Output layer
    ])

    # Compile the autoencoder
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')

    # Train the autoencoder
    autoencoder.fit(scaled_features, scaled_features, epochs=96, batch_size=67, shuffle=True, validation_split=0.1)
    # Reconstruct the data using the autoencoder
    reconstructed_data = autoencoder.predict(scaled_features)

    # Calculate reconstruction errors
    reconstruction_errors = np.mean(np.square(scaled_features - reconstructed_data), axis=1)

    # Set a threshold for anomaly detection
    threshold = np.percentile(reconstruction_errors, 97.5)  # Adjust the percentile as needed
    
    # Identify potential outliers
    potential_outliers = np.where(reconstruction_errors > threshold)[0]

    data['outliers_Autoencoders'] = False
    data.loc[potential_outliers, 'outliers_Autoencoders'] = True

    print(f"Number of detected Potential outliers: {len(potential_outliers)}")
    
    data_clean=data[~data['outliers_Autoencoders']][list_features+['target']]
    print(data_clean.shape)
    print(f"RMSE Score after without the identified outliers is {score_check_unbiased(data_clean,data_pure[list_features+['target']],250)}")
    
    return data
data=autoencoder_anomaly(main_data,data)

### Hyper Parameter tuning

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def objective(trial, main_data, data_pure, list_features):
    global input_dim
    data = main_data.copy()
    
    percentile=trial.suggest_float('percentile', 80, 100)
    encoding_dim = 32  
    epochs = 96
    batch_size = 67

    features = main_data.drop(columns=['id'])

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    input_dim = scaled_features.shape[1]

    autoencoder = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(encoding_dim, activation='relu'),
        layers.Dense(input_dim, activation='linear')  # Output layer
    ])
    


    # Compile the autoencoder
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')

    # Train the autoencoder
    autoencoder.fit(scaled_features, scaled_features, epochs=epochs, batch_size=batch_size, shuffle=True, validation_split=0.1,verbose=False)

    # Reconstruct the data using the autoencoder
    reconstructed_data = autoencoder.predict(scaled_features)

    # Calculate reconstruction errors
    reconstruction_errors = np.mean(np.square(scaled_features - reconstructed_data), axis=1)

    # Set a threshold for anomaly detection
    threshold = np.percentile(reconstruction_errors, percentile)  # Adjust the percentile as needed

    # Identify potential outliers
    potential_outliers = np.where(reconstruction_errors > threshold)[0]

    data['outliers_Autoencoders'] = False
    data.loc[potential_outliers, 'outliers_Autoencoders'] = True

    # Compute the score using score_check_unbiased
    data_clean = data[~data['outliers_Autoencoders']][list_features + ['target']]
    score = score_check_unbiased(data_clean, data_pure[list_features + ['target']], 25)

    return score

# # Create an Optuna study for minimization
# study = optuna.create_study(direction='minimize')

# # Optimize hyperparameters
# study.optimize(lambda trial: objective(trial, main_data, data_pure, list_features), n_trials=100)

# # Get the best hyperparameters
# best_params = study.best_params
# best_percent_cutoff= best_params['percentile']

# 4.4 Local Outlier Factor(LOF)

<font size="3">**Local Outlier Factor (LOF)** is an anomaly detection algorithm that quantifies the local deviation of a data point with respect to its neighbors. It's particularly useful for identifying outliers in datasets where the density of data points varies across different regions</font>

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

params={'n_neighbors': 8, 'contamination': 0.01019797151100069}

def lof(data_pure,data):
    features = data_pure.drop('id', axis=1)
    lof = LocalOutlierFactor(**params)  # Adjust contamination based on your data
    anomalies = lof.fit_predict(features)  # Negative scores are outliers

    outliers = anomalies == -1

    # Combine the outlier information with the original data and labels
    data['outliers_LOF'] = outliers

    # Print the identified outliers
    identified_outliers = data[data['outliers_LOF']]
    print(f"Number of detected Potential outliers: {identified_outliers.shape[0]}")
    
    data_clean=data[~data['outliers_LOF']][list_features+['target']]
    print(data_clean.shape)
    print(f"RMSE Score after without the identified outliers is {score_check_unbiased(data_clean,data_pure[list_features+['target']],250)}")
    
    return data
data=lof(data_pure,data)

### Hyperparameter Tuning

In [ ]:
# Define the objective function for hyperparameter optimization
def objective(trial, main_data, data_pure, list_features):
    data = data_pure.copy()

    # Define hyperparameters to tune
    n_neighbors = trial.suggest_int('n_neighbors', 3, 20)  # Adjust this based on your data
    contamination = trial.suggest_float('contamination', 0.01, 0.5)

    model = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
    predicted_labels = model.fit_predict(main_data.drop(columns=['id']))

    # Identify potential outliers
    outliers = predicted_labels == -1

    data['outlier_LOF'] = outliers

    # Compute the score using score_check_unbiased
    data_clean = data[~data['outlier_LOF']][list_features + ['target']]
    score = score_check_unbiased(data_clean, data_pure[list_features + ['target']], 25)

    return score

# # Create an Optuna study for minimization
# study = optuna.create_study(direction='minimize')

# # Optimize hyperparameters
# study.optimize(lambda trial: objective(trial, main_data, data_pure, list_features), n_trials=100)

# # Get the best hyperparameters
# best_params = study.best_params
# best_n_neighbors = best_params['n_neighbors']
# best_contamination = best_params['contamination']

# # Update the LOF model with the best hyperparameters
# best_model = LocalOutlierFactor(n_neighbors=best_n_neighbors, contamination=best_contamination)
# predicted_labels = best_model.fit_predict(main_data.drop(columns=['id']))

# # Rest of your code here, using the best_model


# 4.5 PCA

In [ ]:
def pca_anamolies(main_data, data):
    features = main_data.drop('id', axis=1)

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    pca = PCA(n_components=2)  # Choose the number of components for visualization
    principal_components = pca.fit_transform(scaled_features)

    # Calculate the reconstruction error (MSE) for each data point
    reconstruction_errors = ((scaled_features - pca.inverse_transform(principal_components)) ** 2).mean(axis=1)

    # Set a threshold for anomaly detection
    threshold = 4 # Adjust the threshold based on your data and desired sensitivity

    # Identify potential outliers
    potential_outliers = [index for index, error in enumerate(reconstruction_errors) if error > threshold]

    # Create a new column 'outliers' in the DataFrame
    data['outliers_PCA'] = False
    data.loc[potential_outliers, 'outliers_PCA'] = True

    print(f"Number of detected Potential outliers: {len(potential_outliers)}")
    
    data_clean=data[~data['outliers_PCA']][list_features+['target']]
    print(data_clean.shape)
    print(f"RMSE Score after without the identified outliers is {score_check_unbiased(data_clean,data_pure[list_features+['target']], 250)}")

    # Plot the data with potential outliers highlighted
    plt.scatter(principal_components[:, 0], principal_components[:, 1], c='green', label='Normal Data')
    plt.scatter(principal_components[potential_outliers, 0], principal_components[potential_outliers, 1], c='red', label='Potential Outliers')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend()
    plt.title('PCA with Potential Outliers')
    plt.show()
    return data
data=pca_anamolies(main_data, data)

**CONCLUSION**

<font size="3">Let's drop if any of the methods flag an outlier</font>

In [ ]:
data['Vote']=data[["outlier_One_SVM","outlier_ISF","outliers_Autoencoders","outliers_LOF","outliers_PCA"]].apply(lambda row: True if row.sum() >=4 else False, axis=1)
# data['Vote']=np.where(data['target']>18,False,data['Vote'])
# data['Or_gate']=  data['outliers_PCA'] | (data['Vote'])

normal_data=data[~data["Vote"]]
print(f"{data.shape[0]-normal_data.shape[0]} datapoints are identfied as outliers")

In [ ]:
data=main_data.loc[normal_data.index]

## Bad Data Points!

<font size="3"> Few data cleaning discoveries have been adopted from interesting findings by other notebooks</font>

* Clip Target data with bounds (7, 20), mentioned in the notebook and one bad data point [yaaangzhou](https://www.kaggle.com/code/yaaangzhou/playground-s3-e21-features-selection-and-tricks/notebook).
* Additional datapoints are removed based on the notebook [warcoder](https://www.kaggle.com/code/warcoder/lb-1-32253-lof-svm-iforest-cleanlab)

In [ ]:
print(f"cleaned data has {data.shape[0]} rows and a CV RMSE score {score_check_unbiased(data,data_pure,100)} without removing bad datapoints")
bad_data_points=[2365, 1089, 1936, 1680, 211,2294,448, 437,309,1684]
for i in bad_data_points:
    if i in data.index:
        if score_check_unbiased(data,data_pure,100)>score_check_unbiased(data.drop(i),data_pure,100):
            data=data.drop(i)
            print(f"Bad Data Point {i} is removed based on CV Check")
        else:
            print(f"No improvement in score by removing the data point {i}")
    else:
        print(f'Bad data point {i} was handle by anomaly detection methods')
        
score=score_check_unbiased(data,data_pure,100)            
print(f"data has {data.shape[0]} rows and a CV RMSE score {score} After removing bad datapoints")


# 5. Feature Importance - without Outliers

<font size="3"> It will be interesting to see the change in feature importance post removal of potential outliers and bad datapoints</font>

In [ ]:
list_features,best_score = get_most_important_features(data, data_pure)
print(list_features)

In [ ]:
post_anomaly_best_features=list_features[:feature_selection(data,data_pure,list_features)]
print(f"Important features after removal of anomalies are {post_anomaly_best_features}")

In [ ]:
print(f"Feature(s) contributing to handle outliers are {set(pre_anomaly_best_features)-set(post_anomaly_best_features)}")
print(f"Additional feature(s) contributing to handle normal datapoints are {set(post_anomaly_best_features)-set(pre_anomaly_best_features)}")

<font size="3">Now, let us consider common features from pre and post processing along with BOD_5 because the test data which is hidden will also have outliers</font>

# 6. Final Feature Selection

In [ ]:
final_features=[set(pre_anomaly_best_features)| set(post_anomaly_best_features)]
zero_features=[f for f in data.columns if f not in final_features+['target']]
for col in zero_features:
    data[col]=0
    
data['id']=0 # It doesn't make sense to keep this at all, ensure that this is 0
data.head()

<font size="3">Now, let's see how the data is distributed</font>

In [ ]:
for col in final_features + ['target']:
    plt.figure(figsize=(10, 4)) 
    plt.plot(data[col], color='green',label=col)  
    plt.title(col)
    plt.xlabel("Index") 
    plt.ylabel("Value")  
    plt.legend()  
    plt.grid(True) 
    plt.tight_layout()  
    plt.show()


**INFERENCES:**

<font size="3"> Without Clipping target, many outliers are handled by anomaly detection techniques</font>

# 7. Clip Target

<font size="3">Only lower bound with 7</font>

In [ ]:
data_new=data.copy()
data_new['target'] = data_new['target'].clip(6.6,20)

In [ ]:
data_new.shape

In [ ]:
data_new.to_csv("submission_moderate.csv")
data_new=data_new.reset_index(drop=True)

# 8. Add Bad Data

In [ ]:
# drop=data_new[(data_new['target']<7) & ((data_new['O2_1']+data_new['O2_2'])/2>7)].index.to_numpy()
# final_data=data_new.drop(drop)

In [ ]:
'''From submission from my other notebook to fill rows upto 3500'''
sub=pd.read_csv("/kaggle/input/arithmetic-mean-power-of-simple-math/submission.csv")

sub_3450=sub.iloc[:3450,:]

In [ ]:
from collections import OrderedDict
from sklearn.model_selection import cross_val_predict

def compute_rmse_contributions(X, y, n_estimators=1000, max_depth=7, random_state=42):
    """
    Compute the RMSE contribution for each data point using cross-validation.
    """

    model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=random_state)

    # Calculate RMSE for each data point using cross-validation
    predicted_y = cross_val_predict(model, X, y, cv=5)  # You can adjust cv as needed

    # Calculate RMSE for each data point
    rmse_contributions = []
    rmse=[]
    for i in range(len(X)):
        y_true_i = y.iloc[i] if isinstance(y, pd.Series) else y[i]
        y_pred_i = predicted_y[i]
        rmse_i = np.sqrt(mean_squared_error([y_true_i], [y_pred_i]))
        rmse_contributions.append((i, rmse_i))
        rmse.append(rmse_i)
    print(np.mean(rmse))

    # Sort RMSE contributions in decreasing order
    sorted_rmse_contributions = OrderedDict(sorted(rmse_contributions, key=lambda x: x[1], reverse=True))

    return sorted_rmse_contributions

X=data_new.drop(columns=['target'])
y=data_new['target']
sorted_rmse_contributions = compute_rmse_contributions(X, y)


In [ ]:
best_indexes = list(sorted_rmse_contributions.keys())[-50:][::-1]
print("Top Indexes with Low RMSE:")
for index in best_indexes:
    rmse = sorted_rmse_contributions[index]
    print(f"Index {index}: RMSE = {rmse}")

In [ ]:
X=sub.drop(columns=['target'])
y=sub['target']
sorted_rmse_contributions = compute_rmse_contributions(X, y)

In [ ]:
best_indexes_sub = list(sorted_rmse_contributions.keys())[-50:][::-1]

print("Top Indexes with Low RMSE:")
for index in best_indexes_sub:
    rmse = sorted_rmse_contributions[index]
    print(f"Index {index}: RMSE = {rmse}")


In [ ]:
sub=sub.drop(best_indexes_sub)
final_data=pd.concat([sub,data_new.loc[best_indexes]], axis="rows")

In [ ]:
# data_new.loc[worst_indexes]

# 9. Submission

In [ ]:
final_data.to_csv("submission.csv",index=False)
print(f"Finally, {final_data.shape[0]} data points are submitted")
final_data.head()

In [ ]:
final_data.describe()